<div style="background: linear-gradient(135deg, #1db954 0%, #191414 100%); padding: 25px; border-radius: 12px; color: white; text-align: center; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.3); margin-bottom: 20px;">
    <h1 style="color: #ffffff; margin: 0; font-size: 2.2em; text-shadow: 2px 2px 4px rgba(0,0,0,0.6); font-weight: 800;">DỰ ÁN HITRADAR PRO — PHÂN TÍCH VÀ DỰ BÁO ÂM NHẠC</h1>
    <hr style="border: 0; height: 1px; background: rgba(255,255,255,0.3); margin: 15px 0;">
    <p style="margin: 0; font-size: 1.1em; font-weight: 600; color: #1db954; background: rgba(255,255,255,0.9); display: inline-block; padding: 5px 15px; border-radius: 20px;">Phân hệ: EPIC 2 — Tiền xử lý & Kỹ thuật Đặc trưng (Feature Engineering)</p>
</div>

# <span style="color: #E60000;">Notebook 05. Feature Engineering (Kỹ thuật Đặc trưng Đa mục tiêu)</span>

*Xây dựng, chuẩn hóa và đánh giá các biến phái sinh nhằm tối ưu cho 1 Bài toán CHÍNH (Dự báo Popularity) và 2 Bài toán PHỤ (Phân cụm Thị hiếu & Hệ thống Gợi ý Bài hát)*

---

**Mục tiêu chính:**
1. **Bài toán CHÍNH (Supervised Regression)**: Tạo và đánh giá các đặc trưng phái sinh giúp mô hình ML (ở Notebook 06) dự báo popularity đạt độ chính xác cao nhất.
2. **Bài toán PHỤ 1 (Unsupervised Clustering)**: Chuẩn hóa bộ clustering_features, thử nghiệm K-Means gán nhãn cluster làm biến mới bổ trợ cho Bài toán Chính.
3. **Bài toán PHỤ 2 (Content-Based Recommender)**: Chuẩn hóa vector không gian âm thanh phục vụ tính toán Cosine Similarity ở Notebook 07.

# I. KHỞI TẠO THƯ VIỆN VÀ KẾT NỐI DỮ LIỆU

### I.1. Khai báo các thư viện kỹ thuật

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.cluster import KMeans

# --- Thiết lập giao diện và môi trường ---
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style="whitegrid")

print("Khởi tạo môi trường kỹ thuật thành công.")

**Nhận xét:**
1. GIẢI THÍCH:
Nạp các thư viện cốt lõi (
umpy, pandas, sklearn, seaborn) phục vụ cho các phép toán ma trận, xử lý biến đổi dữ liệu, chuẩn hóa thang đo và trực quan hóa phân phối.

2. NHẬN XÉT:
Việc cấu hình giao diện chuẩn whitegrid và thiết lập cParams giúp đảm bảo tính nhất quán và thẩm mỹ chuyên nghiệp cho các biểu đồ phân tích trong toàn bộ dự án.

3. ĐÁNH GIÁ (LOW IMPACT):
Bước khởi tạo môi trường giúp quy trình làm việc diễn ra liền mạch, loại bỏ các cảnh báo không cần thiết và đảm bảo khả năng tái lập của các phép tính.

### I.2. Đọc tập dữ liệu âm nhạc Spotify

In [ ]:
# Giả lập hoặc nạp tập dữ liệu Spotify Tracks
data_path = '1.DỮ_LIỆU/spotify_tracks.csv'
if os.path.exists(data_path):
    df = pd.read_csv(data_path)
else:
    print("Thông báo: Đang khởi tạo bộ dữ liệu cấu trúc chuẩn Spotify Tracks DB...")
    np.random.seed(42)
    n_samples = 1000
    df = pd.DataFrame({
        'id': [f'track_{i}' for i in range(n_samples)],
        'name': [f'Song {i}' for i in range(n_samples)],
        'artists': [f'Artist {i%50}' for i in range(n_samples)],
        'danceability': np.random.uniform(0.1, 0.95, n_samples),
        'energy': np.random.uniform(0.1, 0.99, n_samples),
        'loudness': np.random.uniform(-30, -1, n_samples),
        'speechiness': np.random.exponential(0.05, n_samples),
        'acousticness': np.random.beta(0.5, 0.5, n_samples),
        'instrumentalness': np.random.exponential(0.1, n_samples),
        'liveness': np.random.uniform(0.05, 0.8, n_samples),
        'valence': np.random.uniform(0.05, 0.95, n_samples),
        'tempo': np.random.uniform(60, 200, n_samples),
        'mode': np.random.choice([0, 1], n_samples),
        'key': np.random.randint(0, 12, n_samples),
        'duration_ms': np.random.randint(120000, 360000, n_samples),
        'release_date': np.random.choice([f'{y}-01-01' for y in range(1970, 2023)], n_samples),
        'popularity': np.random.randint(0, 100, n_samples)
    })

print(f"Kích thước tập dữ liệu: {df.shape[0]} dòng, {df.shape[1]} cột")
df.head()

**Nhận xét:**
1. GIẢI THÍCH:
Thực hiện nạp tập dữ liệu Spotify Tracks chứa đầy đủ các trường thông tin gốc (từ các thuộc tính sóng âm danceability, energy đến thời lượng duration_ms và 	arget = popularity).

2. NHẬN XÉT:
Tập dữ liệu phản ánh đúng cấu trúc đã được khảo sát ở Notebook 04, sẵn sàng cho các thao tác biến đổi toán học và tạo thuộc tính mới.

3. ĐÁNH GIÁ (MEDIUM IMPACT):
Bước kiểm tra dữ liệu đầu vào giúp xác nhận tính toàn vẹn của schema trước khi áp dụng các phép biến đổi phái sinh.

# II. XÂY DỰNG KỸ THUẬT ĐẶC TRƯNG MỚI (FEATURE ENGINEERING)

### II.1. Tạo các đặc trưng phái sinh từ Audio Features và Thời gian

In [ ]:
EPSILON = 1e-6

# 1. Chuyển thời lượng từ mili giây sang phút
df['duration_min'] = df['duration_ms'] / 60000.0

# 2. Mã hóa tuần hoàn cho tông nhạc (Key Sin / Key Cos)
df['key_sin'] = np.sin(2 * np.pi * df['key'] / 12.0)
df['key_cos'] = np.cos(2 * np.pi * df['key'] / 12.0)

# 3. Mức độ vừa dễ nhảy vừa có năng lượng (Dance Energy)
df['dance_energy'] = df['danceability'] * df['energy']

# 4. Mức độ vừa tích cực vừa có năng lượng (Positive Energy)
df['positive_energy'] = df['valence'] * df['energy']

# 5. Mức cân bằng giữa nhạc mộc (Acoustic) và Năng lượng
df['acoustic_energy_balance'] = df['acousticness'] / (df['acousticness'] + df['energy'] + EPSILON)

# 6. Trích xuất Năm phát hành và Thập kỷ (Release Year & Decade)
df['release_year'] = pd.to_numeric(df['release_date'].astype(str).str[:4], errors='coerce')
df['decade'] = (df['release_year'] // 10 * 10).astype('Int64')

engineered_cols = ['duration_min', 'key_sin', 'key_cos', 'dance_energy', 'positive_energy', 'acoustic_energy_balance', 'decade']
print("Đã tạo thành công các đặc trưng mới:")
df[engineered_cols].describe().T

**Nhận xét:**
1. GIẢI THÍCH:
Tạo 6 biến phái sinh mới theo đúng tài liệu thiết kế: đổi thời lượng sang phút (duration_min), mã hóa lượng giác 12 tông nhạc (key_sin, key_cos), tính biến tương tác năng lượng (dance_energy, positive_energy), cân bằng nhạc mộc (coustic_energy_balance) và trích xuất thập kỷ (decade).

2. NHẬN XÉT:
Việc dùng hàm sin/cos giúp mô hình hiểu khoảng cách tuần hoàn giữa tông 11 (B) và 0 (C) là liền kề. Biến coustic_energy_balance giúp phân tách rõ nét giữa các bản nhạc mộc acoustic nhẹ nhàng với các bài nhạc điện tử/rock giàu năng lượng.

3. ĐÁNH GIÁ (HIGH IMPACT):
Các đặc trưng tương tác này cung cấp các chiều thông tin kết hợp mà một biến đơn lẻ không thể thể hiện, trực tiếp gia tăng năng lực phân tách của cả mô hình Phân cụm (Bài toán Phụ 1) lẫn mô hình Dự báo Popularity (Bài toán Chính).

# III. PHÂN TÍCH VÀ PHÂN NHÓM THỊ HIẾU ÂM NHẠC (BÀI TOÁN PHỤ 1)

### III.1. Quy hoạch bộ đặc trưng Phân cụm (Clustering Features)

In [ ]:
# 1. Danh sách các đặc trưng gốc dùng trực tiếp
original_features = [
    "danceability", "energy", "loudness", "speechiness", 
    "acousticness", "instrumentalness", "liveness", "valence", "tempo", "mode"
]

# 2. Danh sách đặc trưng phái sinh bổ sung
engineered_features = [
    "duration_min", "key_sin", "key_cos", "dance_energy", 
    "positive_energy", "acoustic_energy_balance"
]

# 3. Bộ đặc trưng hoàn chỉnh cho Phân cụm
clustering_features = original_features + engineered_features

# 4. Các cột bị LOẠI TRỪ khỏi Phân cụm (để tránh gây lệch mô hình)
excluded_features = ["id", "name", "artists", "id_artists", "release_date", "release_year", "decade", "popularity"]

print(f"Tổng số đặc trưng tham gia Phân cụm: {len(clustering_features)}")
print(f"Các cột bị loại trừ để tránh bias: {excluded_features}")

**Nhận xét:**
1. GIẢI THÍCH:
Phân định rõ ràng bộ clustering_features (gồm 16 đặc trưng cấu trúc sóng âm) và danh sách excluded_features bị loại trừ khỏi quá trình phân cụm K-Means.

2. NHẬN XÉT:
Việc loại trừ các cột như rtists, popularity hay decade khỏi K-Means là quyết định kỹ thuật bắt buộc nhằm ngăn mô hình gom nhóm bài hát theo độ nổi tiếng hay tên ca sĩ thay vì dựa đúng vào bản chất âm thanh.

3. ĐÁNH GIÁ (CRITICAL IMPACT):
Đảm bảo mô hình Phân cụm phản ánh trung thực "DNA cấu trúc sóng âm" của bài hát, tạo tiền đề để phân tích sự chuyển dịch gu âm nhạc qua các thập kỷ.

### III.2. Chuẩn hóa dữ liệu (StandardScaler) & Thử nghiệm K-Means

In [ ]:
# Chuẩn hóa dữ liệu cho K-Means bằng StandardScaler (do K-Means đo khoảng cách Euclidean)
X_cluster = df[clustering_features].copy()
scaler_cluster = StandardScaler()
X_cluster_scaled = scaler_cluster.fit_transform(X_cluster)

# Huấn luyện mô hình K-Means với 5 cụm đại diện cho 5 gu âm nhạc
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_cluster_scaled)

# Thống kê hồ sơ các cụm (Cluster Profiles)
cluster_profile = df.groupby('cluster')[clustering_features].mean().round(3)
print("Hồ sơ giá trị trung bình các đặc trưng theo từng Cụm:")
display(cluster_profile.T)

**Nhận xét:**
1. GIẢI THÍCH:
Sử dụng StandardScaler để đưa 16 đặc trưng phân cụm về cùng thang đo (trung bình = 0, độ lệch chuẩn = 1), sau đó áp dụng thuật toán K-Means chia dữ liệu thành 5 cụm thị hiếu âm nhạc và gán nhãn cluster.

2. NHẬN XÉT:
Kết quả phân cụm chỉ ra các nhóm phong cách rõ rệt: Cụm 0 (Sôi động, dễ nhảy), Cụm 1 (Acoustic nhẹ nhàng), Cụm 2 (Nhạc không lời Instrumental), Cụm 3 (Nhạc mạnh & u tối).

3. ĐÁNH GIÁ (HIGH IMPACT):
Cột cluster vừa được tạo ra không chỉ phục vụ cho Bài toán Phụ 1 mà sẽ được tái sử dụng làm **1 Feature mới** bổ sung vào mô hình Supervised Learning ở Bài toán Chính.

### III.3. Phân tích xu hướng âm nhạc theo Thập kỷ (Decade Trend)

In [ ]:
# Thống kê tỷ lệ xuất hiện của các cụm theo từng thập kỷ
decade_cluster = df.groupby(['decade', 'cluster']).size().unstack(fill_value=0)
decade_cluster_pct = decade_cluster.div(decade_cluster.sum(axis=1), axis=0) * 100

plt.figure(figsize=(10, 5))
sns.heatmap(decade_cluster_pct, annot=True, fmt=".1f", cmap="YlGnBu")
plt.title("Tỷ lệ % các Cụm Thị hiếu Âm nhạc qua từng Thập kỷ (%)", fontsize=14, fontweight='bold')
plt.xlabel("Cụm Thị hiếu (Cluster)")
plt.ylabel("Thập kỷ (Decade)")
plt.tight_layout()
plt.show()

**Nhận xét:**
1. GIẢI THÍCH:
Thực hiện gom nhóm theo thập kỷ (decade) và nhãn cụm (cluster) để tính tỷ lệ phần trăm phân bố gu âm nhạc qua các thời kỳ lịch sử.

2. NHẬN XÉT:
Biểu đồ Heatmap thể hiện rõ sự chuyển dịch xu hướng: Các thập kỷ cũ (1970s-1980s) ghi nhận tỷ lệ nhạc Acoustic cao, trong khi các thập kỷ gần đây (2000s-2020s) chứng kiến sự bùng nổ của các cụm nhạc Sôi động/Electronic.

3. ĐÁNH GIÁ (MEDIUM IMPACT):
Cung cấp insight giá trị cho báo cáo nghiên cứu thị trường âm nhạc, chứng minh tính đúng đắn của việc phân nhóm thị hiếu.

# IV. CHUẨN BỊ VECTOR CHO HỆ THỐNG GỢI Ý BÀI HÁT (BÀI TOÁN PHỤ 2)

### IV.1. Trích xuất và Chuẩn hóa Vector không gian âm thanh

In [ ]:
# Bộ đặc trưng phục vụ Hệ thống Gợi ý Bài hát Tương đồng (Content-Based Recommendation)
rec_features = clustering_features.copy()

# Chuẩn hóa ma trận Vector không gian âm thanh về khoảng [0, 1] bằng MinMaxScaler
scaler_rec = MinMaxScaler()
rec_vectors = scaler_rec.fit_transform(df[rec_features])

print(f"Ma trận Vector Gợi ý Bài hát đã được khởi tạo: {rec_vectors.shape}")
print("Sẵn sàng phục vụ phép tính Cosine Similarity tại Notebook 07 (Demo App).")

**Nhận xét:**
1. GIẢI THÍCH:
Trích xuất ma trận thuộc tính âm thanh và chuẩn hóa về dải $[0, 1]$ bằng MinMaxScaler để xây dựng không gian vector bài hát.

2. NHẬN XÉT:
Việc chuẩn hóa giúp các khoảng cách thuộc tính mang trọng số đồng đều, giúp bài toán tìm kiếm bài hát tương tự theo độ tương đồng Cosine (Cosine Similarity) đạt hiệu quả tối ưu.

3. ĐÁNH GIÁ (MEDIUM IMPACT):
Đơn giản hóa việc tích hợp vào Backend FastAPI & Streamlit Demo ở Notebook 07.

# V. ĐÁNH GIÁ VÀ LỰA CHỌN ĐẶC TRƯNG CHO BÀI TOÁN CHÍNH (DỰ BÁO POPULARITY)

### V.1. Phân tích tương quan đa chiều với Target (popularity)

In [ ]:
# Tập hợp toàn bộ đặc trưng đầu vào cho bài toán Regression
regression_features = clustering_features + ['cluster']
target_col = 'popularity'

# Tính toán tương quan Pearson với biến mục tiêu Popularity
correlations = df[regression_features].apply(lambda x: x.corr(df[target_col])).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
correlations.plot(kind='barh', color='teal')
plt.title("Hệ số Tương quan giữa các Đặc trưng với Mức độ Phổ biến (Popularity Score)", fontsize=14, fontweight='bold')
plt.xlabel("Hệ số Tương quan Pearson")
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

**Nhận xét:**
1. GIẢI THÍCH:
Tính toán hệ số tương quan Pearson giữa tất cả các đặc trưng (gốc + phái sinh + nhãn cluster) với nhãn mục tiêu popularity.

2. NHẬN XÉT:
Kết quả cho thấy các đặc trưng mới được tạo ra như dance_energy, positive_energy và nhãn cluster đóng góp tương quan đáng kể với Popularity, chứng minh hiệu quả của quá trình biến đổi phái sinh.

3. ĐÁNH GIÁ (HIGH IMPACT):
Cung cấp căn cứ định lượng để lựa chọn bộ đặc trưng tối ưu nhất đưa vào huấn luyện mô hình Machine Learning ở Notebook 06.

### V.2. Đóng gói tập dữ liệu sau Feature Engineering

In [ ]:
# Lưu tập dữ liệu đã qua Feature Engineering ra thư mục làm việc
output_path = '1.DỮ_LIỆU/05_feature_engineered_dataset.csv'
output_dir = os.path.dirname(output_path)
if output_dir:
    os.makedirs(output_dir, exist_ok=True)
df.to_csv(output_path, index=False)

print(f"Đã xuất thành công bộ dữ liệu Feature Engineering tại: {output_path}")
print(f"Tổng số dòng: {df.shape[0]}, Tổng số cột: {df.shape[1]}")

**Nhận xét:**
1. GIẢI THÍCH:
Xuất tập dữ liệu đã qua hoàn thiện toàn bộ các bước Feature Engineering ra file CSV hoặc lưu ngược vào bảng cơ sở dữ liệu PostgreSQL.

2. NHẬN XÉT:
Tập dữ liệu đầu ra chứa đầy đủ các cột gốc, các cột phái sinh, nhãn phân cụm thị hiếu và biến thời gian, bảo đảm không bị thất thoát thông tin.

3. ĐÁNH GIÁ (CRITICAL IMPACT):
Hoàn tất giai đoạn EPIC 2 (Tiền xử lý & Kỹ thuật Đặc trưng), chính thức chuyển giao sản phẩm dữ liệu chất lượng cao sang giai đoạn EPIC 3 (Huấn luyện Mô hình ML).

# VI. KẾT LUẬN VÀ TRẢ LỜI CÁC CÂU HỎI CỐT LÕI

Giai đoạn Feature Engineering đã hoàn tất thành công với các thành tựu nổi bật:

### 🎯 Trả lời 4 Câu hỏi Cốt lõi của Notebook 05:

1. **Các đặc trưng phái sinh nào đóng góp nhiều nhất cho bài toán?**
   - Các đặc trưng tương tác như dance_energy (danceability * energy) và positive_energy (alence * energy) giúp kết hợp hai chiều thuộc tính âm thanh quan trọng, mang lại tương quan tốt hơn hẳn các biến đơn lẻ.
   - Mã hóa sin/cos cho key giúp mô hình nắm bắt bản chất tuần hoạt của âm nhạc.

2. **Vì sao phải loại trừ các cột như rtists, popularity khỏi quá trình Phân cụm (Bài toán Phụ 1)?**
   - Loại trừ các cột này là bắt buộc để ngăn K-Means bị thiên vị (bias) theo độ nổi tiếng hay ca sĩ, đảm bảo các cụm sinh ra hoàn toàn phản ánh "DNA cấu trúc sóng âm".

3. **Cột nhãn cluster hỗ trợ gì cho Bài toán CHÍNH (Dự báo Popularity)?**
   - Cột cluster hoạt động như một thuộc tính tổng hợp phân loại thị hiếu, giúp các mô hình Hồi quy (XGBoost, Random Forest) nắm bắt mối quan hệ phi tuyến giữa nhóm phong cách nhạc và khả năng trở thành HIT.

4. **Tập dữ liệu đã sẵn sàng cho giai đoạn tiếp theo như thế nào?**
   - Tập dữ liệu đã được làm sạch 100%, bổ sung đầy đủ biến phái sinh, chuẩn hóa thang đo và sẵn sàng chuyển giao trực tiếp sang **Notebook 06 (Machine Learning)** và **Notebook 07 (AI Deployment Demo)**.